In [ ]:
from IPython.display import clear_output           # utility that can clear the current cell output

%pip install kagglehub catboost lightgbm tqdm -q   # %pip is an IPython magic command, prefered over !pip install command. -q (quiet mode)

clear_output()                                     # Removes all output produced by the cell. Commonly used after installations to: Hide warnings, Remove progress bars, Keep the notebook clean

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns              # High-level visualization library built on top of matplotlib
import kagglehub
import os
from tqdm import tqdm              # Adds progress bars to loops, Especially useful for long-running tasks (training, downloads, preprocessing)

%matplotlib inline
# Tells Jupyter to render plots directly inside the notebook, Without this, plots may open in separate windows or not appear

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)
#df = df.drop(columns="Unnamed: 0", axis=1) # there was a column with no header so pandas names it unnamed, we need to drop this column when it happens


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
# 1. What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Order_ID", axis=1)#.astype(float)

In [ ]:
# Task 2: Write your code here:
# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()            # df.isnull() → Returns a DataFrame of the same shape with True where the value is NaN
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])     # Print missing columns only that has at least one missing value
  if missing_values.any():                      # Check if any missing values exist overall
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

# Select relevant columns, we do not select 'model' (too many unique values, too sparse and will hurt the model performance)
cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time']
df_clean = df[cols].copy()

# Drop rows where target (Delivery_Time) or key features are missing - can't predict without them
print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=cols)
print(f"After dropping missing missing rows: {df_clean.shape}")

In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
# 3. Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns     # return which columns in my dataset are categorical (returns the columns names)

print("Categorical Columns:", categorical_cols)               # print it as list for cleaner output

categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time', 'Vehicle_Type']

from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["number"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET. include=["int64", "float64"] we can just put include=["number"]
print("Numerical Columns:", numerical_cols)

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# Task 6: Write your code here:
# 1. Is the target imbalanced? i guess we only check if our data is impalanced in classification **make sure
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)     # axis=1 means drop a column (not a row). .astype(float)-> Converts all remaining feature columns to float
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score



# Define Model
model = RandomForestRegressor(n_estimators=200)

# StratifiedKFold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mse_scores = []   # Used to store metrics from each fold
mae_scores = []

for train_idx, test_idx in kf.split(X):                   # Each iteration gives: train_idx → indices for training, test_idx → indices for testing. Ex: train_idx = [0, 1, 2, 5, 6], test_idx = [3, 4]
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mae_scores.append(mean_absolute_error(y_test, y_pred))


# Print Evaluation Metrics
print("\nModel Evaluation Metrics (StratifiedKFold)\n" + "-"*40)
print(f"MSE : {np.mean(mse_scores):.2f}")
print(f"MAE : {np.mean(mae_scores):.2f}")
print(f"RMSE: {np.sqrt(np.mean(mse_scores)):.2f}")
print("-"*40)





# kf = KFold(n_splits=5, shuffle=True, random_state=42)

# for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
#   print(f"\nFold {fold_idx + 1}/{n_splits}")

#   X_train, X_test = X.iloc[train_index], X.iloc[test_index]
#   y_train, y_test = y.iloc[train_index], y.iloc[test_index]

#   model = RandomForestRegressor(n_estimators=200)

#   # Train
#   model.fit(X_train, y_train)

#   # Predict
#   y_pred = model.predict(X_test)

#   # Calculate metrics
#   mse = sklearn_mse(y_test, y_pred)
#   rmse = np.sqrt(mse)
#   r2 = r2_score(y_test, y_pred)

#   # Store results
#   all_results[model_name]["mse"].append(mse)
#   all_results[model_name]["rmse"].append(rmse)
#   all_results[model_name]["r2"].append(r2)

In [ ]:
from sklearn.linear_model import Ridge, Lasso

Ridge_Regression = Ridge(alpha=1.0, max_iter=10000),
LASSO_Regression = Lasso(alpha=1.0,  max_iter=10000),

coeffs = {}    # Extract coefficients (weights) from trained models

coeffs['Lasso'] = LASSO_Regression.coef_
coeffs['Ridge'] = Ridge_Regression.coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))    # Create 1 row, 2 columns of plots: Left → LASSO, Right → Ridge.
axes = axes.flatten()                              # flatten() → makes axes easy to index (axes[0], axes[1])
features = X.columns                               # Store feature names (used as y-axis labels)

for i, (model, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)            # importance magnitude
  sorted_idx = np.argsort(absolute_coef)  # indices that sort values from small → large. sorting features by importance

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])   # feature names, coefficient values
  ax.set_title(f"{model} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
# Task 5: Write your code here:
# 1. What does our target variable (charges) look like?

y_pred.hist(bins=30, edgecolor='black')

plt.title(f"Predict Distribution ")
plt.xlabel(y_pred)
plt.ylabel("Frequency")
plt.grid(False)

plt.show()



In [ ]:
# Task Bonus: Write your code here: